In [ ]:
import pandas as pd
import numpy as np
import gc
import os

INPUT_DATA_PATH = "/kaggle/input/notebooks/luminhanh/ba-model-prep-data/" 

print("🚀 Đang nạp dữ liệu đã tiền xử lý (Phiên bản tối ưu RAM cho Tuning)...")

X_train = pd.read_parquet(INPUT_DATA_PATH + 'X_train.parquet')
X_val = pd.read_parquet(INPUT_DATA_PATH + 'X_val.parquet')

y_train = pd.read_parquet(INPUT_DATA_PATH + 'y_train.parquet').values
y_val = pd.read_parquet(INPUT_DATA_PATH + 'y_val.parquet').values

# Bỏ qua test_id_matrix vì Optuna không cần nộp bài

print(f"✅ Nạp xong! Kích thước X_train: {X_train.shape}")
print(f"✅ Kích thước X_val: {X_val.shape}")

# Dọn rác bộ nhớ ngay lập tức
gc.collect()

🚀 Đang nạp dữ liệu đã tiền xử lý (Phiên bản tối ưu RAM cho Tuning)...
✅ Nạp xong! Kích thước X_train: (1005090, 633)
✅ Kích thước X_val: (167515, 633)


0

In [ ]:
import optuna
from catboost import CatBoostRegressor, Pool
import numpy as np
import pandas as pd
import gc
import json
import os

print("🚀 BƯỚC 1: Chuẩn bị dữ liệu cho CatBoost...")

# 1. Ép kiểu Categorical sang String
cat_features = ['store_nbr', 'item_nbr', 'family', 'class', 'city', 'state', 'type', 'cluster']
for col in cat_features:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype(str)
        X_val[col] = X_val[col].astype(str)

actual_cat_features = [c for c in cat_features if c in X_train.columns]

# 2. Cắt tỉa khớp dòng
train_size = min(X_train.shape[0], y_train.shape[0])
val_size = min(X_val.shape[0], y_val.shape[0])

X_train_fix = X_train.iloc[:train_size].copy()
y_train_fix = np.array(y_train)[:train_size]
X_val_fix = X_val.iloc[:val_size].copy()
y_val_fix = np.array(y_val)[:val_size]

# 3. Trích xuất trọng số
if 'perishable' in X_train_fix.columns:
    train_weights = X_train_fix['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0).values
    val_weights = X_val_fix['perishable'].apply(lambda x: 1.25 if x == 1 else 1.0).values
    
    X_train_model = X_train_fix.drop(columns=['perishable'])
    X_val_model = X_val_fix.drop(columns=['perishable'])
else:
    train_weights = None
    val_weights = None
    X_train_model = X_train_fix
    X_val_model = X_val_fix

del X_train_fix, X_val_fix
gc.collect()

print("🚀 BƯỚC 2: Nạp dữ liệu vào cấu trúc Pool...")

# 4. Lưu trữ 3 Pools (Ngày 1, 8, 16)
target_days = [0, 7, 15] 
pools = {}

for d_idx in target_days:
    train_p = Pool(data=X_train_model, label=y_train_fix[:, d_idx], weight=train_weights, cat_features=actual_cat_features)
    val_p = Pool(data=X_val_model, label=y_val_fix[:, d_idx], weight=val_weights, cat_features=actual_cat_features)
    pools[d_idx] = {'train': train_p, 'val': val_p}

# 5. Hàm mục tiêu với Không gian tìm kiếm RỘNG
def objective(trial):
    param = {
        'iterations': 1200, 
        'loss_function': 'RMSE',
        'eval_metric': 'RMSE',
        'random_seed': 42,
        'task_type': 'GPU',
        'thread_count': 4,
        
        'depth': trial.suggest_int('depth', 6, 10), # Cho phép cây sâu tới 10
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 20.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_categorical('border_count', [128, 254])
    }
    
    scores = []
    
    for step, d_idx in enumerate(target_days):
        model = CatBoostRegressor(**param)
        
        model.fit(
            pools[d_idx]['train'],
            eval_set=pools[d_idx]['val'],
            early_stopping_rounds=30, # Dừng sớm nếu không cải thiện
            verbose=False
        )
        
        score = model.get_best_score()['validation']['RMSE']
        scores.append(score)
        del model
        
        # Báo cáo điểm để Pruning
        trial.report(np.mean(scores), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
            
    gc.collect()
    return np.mean(scores)

# 6. Chạy Optuna
print("🚀 BƯỚC 3: Bắt đầu dò tìm tham số (Giới hạn an toàn: 8.5 tiếng)...")

pruner = optuna.pruners.MedianPruner(n_warmup_steps=0, n_startup_trials=5)
study = optuna.create_study(direction='minimize', study_name='CatBoost_Favorita_MaxOpt', pruner=pruner)

# timeout=30600
study.optimize(objective, n_trials=300, timeout=30600, gc_after_trial=True)

print("\n🏆 QUÁ TRÌNH TỐI ƯU HOÀN TẤT!")
print(f"Tổng số trial đã chạy: {len(study.trials)}")
print("Giá trị RMSE trung bình tốt nhất: ", study.best_value)

print("Bộ tham số Vàng:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value},")

# --- 7. LƯU THAM SỐ RA FILE JSON ---
print("\n💾 Đang lưu bộ tham số ra file...")
OUTPUT_PARAM_FILE = '/kaggle/working/best_catboost_params.json'

with open(OUTPUT_PARAM_FILE, 'w') as f:
    json.dump(study.best_params, f)

print(f"✅ THÀNH CÔNG! File tham số đã được lưu tại: {OUTPUT_PARAM_FILE}")

del pools
gc.collect()

🚀 BƯỚC 1: Chuẩn bị dữ liệu cho CatBoost...
🚀 BƯỚC 2: Nạp dữ liệu vào cấu trúc Pool...


[I 2026-05-17 15:02:28,313] A new study created in memory with name: CatBoost_Favorita_MaxOpt


🚀 BƯỚC 3: Bắt đầu dò tìm tham số (Giới hạn an toàn: 8.5 tiếng)...


[I 2026-05-17 15:12:02,099] Trial 0 finished with value: 0.5841093897696381 and parameters: {'depth': 7, 'learning_rate': 0.010407971012655115, 'l2_leaf_reg': 11.790104475300252, 'random_strength': 5.933348882533956, 'bagging_temperature': 0.05288919870044917, 'border_count': 254}. Best is trial 0 with value: 0.5841093897696381.
[I 2026-05-17 15:26:56,265] Trial 1 finished with value: 0.5797203578270168 and parameters: {'depth': 10, 'learning_rate': 0.011950535848027978, 'l2_leaf_reg': 13.429516137945223, 'random_strength': 4.559628488929244, 'bagging_temperature': 0.4385157590187051, 'border_count': 128}. Best is trial 1 with value: 0.5797203578270168.
[I 2026-05-17 15:42:41,787] Trial 2 finished with value: 0.5819021850447254 and parameters: {'depth': 10, 'learning_rate': 0.008111612163611268, 'l2_leaf_reg': 15.809226555064543, 'random_strength': 3.45783906988563, 'bagging_temperature': 0.5350463220035101, 'border_count': 254}. Best is trial 1 with value: 0.5797203578270168.
[I 2026-


🏆 QUÁ TRÌNH TỐI ƯU HOÀN TẤT!
Tổng số trial đã chạy: 71
Giá trị RMSE trung bình tốt nhất:  0.5734999477051165
Bộ tham số Vàng:
    'depth': 9,
    'learning_rate': 0.059234900214243276,
    'l2_leaf_reg': 3.8258800788689413,
    'random_strength': 4.353721638364459,
    'bagging_temperature': 0.21782504502987693,
    'border_count': 128,

💾 Đang lưu bộ tham số ra file...
✅ THÀNH CÔNG! File tham số đã được lưu tại: /kaggle/working/best_catboost_params.json


32